In [1]:
import os
import random
import numpy as np
import pandas as pd

# suppress protobuf warning from TensorFlow import attempts
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

# Attempt TensorFlow import; on failure we fall back to a sklearn model
try:
    import tensorflow as tf

    TF_AVAILABLE = True
except Exception as e:
    print("TensorFlow import failed, falling back to sklearn:", e)
    TF_AVAILABLE = False

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

IMG_SIZE = 224
NB_CHANNELS = 3
NB_CLASSES = 5  # ratings 0‑4
BATCH_SIZE = 32
TEST_BATCH_SIZE = 32
NUM_WORKERS = min(8, os.cpu_count() or 1)

MODEL_NAME = "resnet50"




2026-05-05 21:02:59.091482: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778014979.104300      55 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778014979.108200      55 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-05 21:02:59.124164: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
# locate the dataset directory among several possible locations
def locate_base():
    candidates = [
        "data/aptos2019-blindness-detection",
        "input/aptos2019-blindness-detection",
        "/kaggle/input/aptos2019-blindness-detection",
        "data",
        "input",
        "/kaggle/input",
    ]
    for c in candidates:
        if os.path.isdir(c) and os.path.isfile(os.path.join(c, "train.csv")):
            return c
    raise FileNotFoundError(
        "Could not locate the aptos2019-blindness-detection dataset directory."
    )


BASE_INPUT = locate_base()

train = pd.read_csv(os.path.join(BASE_INPUT, "train.csv"))
test = pd.read_csv(os.path.join(BASE_INPUT, "test.csv"))

# ensure diagnosis is string for potential TF pipelines (kept for compatibility)
train["diagnosis"] = train["diagnosis"].astype(str)

train["filename"] = train["id_code"] + ".png"
test["filename"] = test["id_code"] + ".png"

print("Number of train samples:", train.shape[0])
print("Number of test samples :", test.shape[0])




Number of train samples: 3295
Number of test samples : 367


In [3]:
def get_image_dir(base_dir: str, subfolder: str) -> str:
    """
    Return the correct directory that contains the images.
    Handles possible nested folder structures.
    """
    candidate = os.path.join(base_dir, subfolder)
    nested = os.path.join(candidate, subfolder)  # e.g. …/train_images/train_images
    if os.path.isdir(nested):
        return nested
    elif os.path.isdir(candidate):
        return candidate
    else:
        raise FileNotFoundError(f"Cannot locate image directory for {subfolder}")


TRAIN_IMG_DIR = get_image_dir(BASE_INPUT, "train_images")
TEST_IMG_DIR = get_image_dir(BASE_INPUT, "test_images")

print("Train images dir:", TRAIN_IMG_DIR)
print("Test images dir :", TEST_IMG_DIR)



Train images dir: /kaggle/input/aptos2019-blindness-detection/train_images/train_images
Test images dir : /kaggle/input/aptos2019-blindness-detection/test_images/test_images


In [4]:
# If TensorFlow is unavailable, train a lightweight sklearn model.
# This avoids the heavy TF pipeline while still providing a usable prediction.
if not TF_AVAILABLE:
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import cohen_kappa_score
    from PIL import Image

    def load_resized_images(df, img_dir, size=(32, 32)):
        """Load images, resize, and flatten to a 2‑D array."""
        imgs = []
        for fname in df["filename"]:
            path = os.path.join(img_dir, fname)
            try:
                img = Image.open(path).convert("RGB").resize(size)
                imgs.append(np.asarray(img).flatten())
            except Exception as e:
                # If an image is missing/corrupt, use a zero array.
                print(f"Warning: could not load {path}: {e}")
                imgs.append(np.zeros(size[0] * size[1] * 3, dtype=np.uint8))
        return np.stack(imgs).astype(np.float32) / 255.0  # normalize

    # Load and preprocess training images
    X = load_resized_images(train, TRAIN_IMG_DIR, size=(32, 32))
    y = train["diagnosis"].astype(int).values

    # Simple train/validation split for local metric estimation
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )

    # Train a RandomForest classifier
    rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        n_jobs=NUM_WORKERS,
        random_state=SEED,
        class_weight="balanced",
    )
    rf.fit(X_train, y_train)

    # Optional: compute local QWK to gauge improvement
    val_pred = rf.predict(X_val)
    qwk = cohen_kappa_score(y_val, val_pred, weights="quadratic")
    print(f"Validation QWK (approximation): {qwk:.5f}")

    # Load test images and predict
    X_test = load_resized_images(test, TEST_IMG_DIR, size=(32, 32))
    predictions = rf.predict(X_test).astype(int)

else:
    # ------ TensorFlow pipeline (kept unchanged for environments where TF works) ------
    from tensorflow.keras.preprocessing.image import ImageDataGenerator
    from tensorflow.keras.applications.resnet import ResNet50, preprocess_input
    from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
    from tensorflow.keras.models import Model

    def get_model(name, input_shape, n_classes):
        """Build a simple transfer‑learning model."""
        if name == "resnet50":
            base = ResNet50(
                weights="imagenet", include_top=False, input_shape=input_shape
            )
            x = GlobalAveragePooling2D()(base.output)
            x = Dropout(0.5)(x)
            output = Dense(n_classes, activation="softmax")(x)
            model = Model(inputs=base.input, outputs=output)

            for layer in base.layers:
                layer.trainable = False
            for layer in base.layers[-10:]:
                layer.trainable = True
            return model
        else:
            raise ValueError(f"Unsupported model name: {name}")

    train_datagen = ImageDataGenerator(
        preprocessing_function=preprocess_input,
        validation_split=0.2,
        horizontal_flip=True,
    )
    val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
    test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

    train_gen = train_datagen.flow_from_dataframe(
        dataframe=train,
        directory=TRAIN_IMG_DIR,
        x_col="filename",
        y_col="diagnosis",
        batch_size=BATCH_SIZE,
        class_mode="categorical",
        target_size=(IMG_SIZE, IMG_SIZE),
        subset="training",
        seed=SEED,
        shuffle=True,
        workers=NUM_WORKERS,
        use_multiprocessing=True,
    )

    val_gen = val_datagen.flow_from_dataframe(
        dataframe=train,
        directory=TRAIN_IMG_DIR,
        x_col="filename",
        y_col="diagnosis",
        batch_size=BATCH_SIZE,
        class_mode="categorical",
        target_size=(IMG_SIZE, IMG_SIZE),
        subset="validation",
        seed=SEED,
        shuffle=False,
        workers=NUM_WORKERS,
        use_multiprocessing=True,
    )

    test_gen = test_datagen.flow_from_dataframe(
        dataframe=test,
        directory=TEST_IMG_DIR,
        x_col="filename",
        batch_size=TEST_BATCH_SIZE,
        class_mode=None,
        target_size=(IMG_SIZE, IMG_SIZE),
        shuffle=False,
        seed=SEED,
        workers=NUM_WORKERS,
        use_multiprocessing=True,
    )

    model = get_model(MODEL_NAME, (IMG_SIZE, IMG_SIZE, NB_CHANNELS), NB_CLASSES)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    EPOCHS = 5
    model.fit(
        train_gen,
        steps_per_epoch=train_gen.n // BATCH_SIZE,
        validation_data=val_gen,
        validation_steps=val_gen.n // BATCH_SIZE,
        epochs=EPOCHS,
        verbose=1,
    )
    test_gen.reset()
    preds = model.predict(test_gen, verbose=1)
    predictions = np.argmax(preds, axis=1).astype(int)



Found 0 validated image filenames belonging to 0 classes.
Found 0 validated image filenames belonging to 0 classes.
Found 0 validated image filenames.


/usr/local/lib/python3.11/dist-packages/keras/src/legacy/preprocessing/image.py:920: UserWarning: Found 3295 invalid image filename(s) in x_col="filename". These filename(s) will be ignored.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/keras/src/legacy/preprocessing/image.py:920: UserWarning: Found 3295 invalid image filename(s) in x_col="filename". These filename(s) will be ignored.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/keras/src/legacy/preprocessing/image.py:920: UserWarning: Found 367 invalid image filename(s) in x_col="filename". These filename(s) will be ignored.
  warnings.warn(
2026-05-05 21:03:06.160377: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1778014986.160851      55 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: 

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


ValueError: The PyDataset has length 0

In [5]:
# Build submission file
ids = test["id_code"].tolist()
submission = pd.DataFrame({"id_code": ids, "diagnosis": predictions})
submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission saved to {submission_path}")
print(submission.head(10))

NameError: name 'predictions' is not defined